# 1 Probabilistic forecasting: the line is not the answer

Every model in `00_forecasting_baselines` output one number per half-hour. Suppose you hand
that to the person scheduling tomorrow's grid dispatch and they ask the question they
actually have:

> *What is the chance demand tops 9000 MW during tomorrow evening's peak?*

You cannot answer. Not because your model is bad — because a point forecast is not
the kind of object that has an answer to that question in it. It is a single line
through a future that has a width.

This notebook gives forecasts a width. It does it in the order that makes the ideas
stick rather than the order the papers were published:

1. a **free** probabilistic baseline, so the trained ones have something to beat;
2. **CRPS**, the metric that scores a distribution — and quietly generalises the MAE
   you already trust;
3. two heads you write yourself: a **Gaussian** head trained with NLL, and a
   **quantile** head trained with pinball loss;
4. **calibration** — is an 80% interval right 80% of the time? (Spoiler: no.)

The series and the windows are those of `00_forecasting_baselines`:
`australian_electricity_demand`, half-hourly grid demand with a clean daily-and-weekly
pattern, four days of history and one day of horizon, scaled on the training statistics.
That notebook named seasonal naive as the point baseline and trained a linear model and a
GRU on these windows; the heads below sit on the same two trunks. Calibration is easiest to
learn to read on a series that behaves.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from australian_electricity import load_ausalia
from windowing import window_index
from notebooktester import param

DAY = 48  # half-hourly steps per day
W, H = (
    4 * DAY,
    DAY,
)  # four days of history, one day of horizon: the window of 00_forecasting_baselines
STRIDE = 24  # 12 hours between training windows (00_forecasting_baselines 0.1)
EPOCHS = param(8, test=1)
torch.manual_seed(0)

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")

## 1.1 The width is real, and it is not your model's fault

Before blaming a model for missing the future, check how much of the future was
knowable. Take a window, find the other windows in the training set whose last 24
hours look almost identical to it, and plot what happened next in each case.

In [ ]:
probe = 400
recent = xtr[:, -DAY:]
distance = (recent - xtr[probe, -DAY:]).pow(2).mean(1)
neighbours = distance.argsort()[:40]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    range(-DAY, 0), xtr[probe, -DAY:], color="k", lw=2, label="the last day we observed"
)
for j in neighbours:
    ax.plot(range(0, H), ytr[j], color="tab:blue", lw=0.7, alpha=0.35)
ax.plot(
    range(0, H), ytr[probe], color="crimson", lw=2, label="what actually happened once"
)
ax.axvline(0, color="0.8")
ax.set_xlabel("half-hours from the forecast origin")
ax.set_ylabel("scaled demand")
ax.set_title("40 near-identical histories, and the 40 different futures that followed")
ax.legend(fontsize=8)
plt.tight_layout()

spread = torch.stack([ytr[j] for j in neighbours])
print(
    f"across these 40 near-identical histories, the outcome at +1 day ranges over "
    f"{float(spread[:, -1].max() - spread[:, -1].min()):.2f} scaled units"
)

Forty histories a model could not tell apart, and forty visibly different futures. No
amount of capacity fixes that: the information simply is not in the window. This is
**aleatoric** uncertainty — irreducible given what you observe — as opposed to the
**epistemic** kind you could reduce with more data or a better model.

A point forecast reports the middle of that fan and says nothing about its width. The
rest of this notebook is about reporting the fan.

## 1.2 CRPS: one number that scores a distribution

Before building anything, decide how you will score it — otherwise you will pick the
model whose pictures you like.

The **Continuous Ranked Probability Score** works on a forecast expressed as *samples*.
Given `S` sample paths and the truth `y`:

$$\mathrm{CRPS} \approx \underbrace{\frac{1}{S}\sum_i |s_i - y|}_{\text{are the samples near the truth?}} - \underbrace{\frac{1}{2S^2}\sum_{i,j} |s_i - s_j|}_{\text{are the samples spread out?}}$$

The first term rewards accuracy, the second penalises hedging. Widening your intervals
until they cover everything makes the first term better and the second worse. Lower is
better, and it is measured in the units of the data.

The property that makes it the right metric to standardise on: **for a forecast with no
spread at all, CRPS is exactly MAE.** The second term vanishes. So CRPS is not a new
scale to learn — it is MAE, extended to forecasts that have a width, and every point
model in `00_forecasting_baselines` can be dropped straight into the same table.

In [ ]:
def crps(y: torch.Tensor, samples: torch.Tensor) -> float:
    # y: (N, H)   samples: (N, S, H)
    s = samples.shape[1]
    accuracy = (samples - y.unsqueeze(1)).abs().mean(1)
    spread = (samples.unsqueeze(1) - samples.unsqueeze(2)).abs().sum((1, 2)) / (
        2 * s * (s - 1)
    )
    return float((accuracy - spread).mean())


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def coverage(y: torch.Tensor, lo: torch.Tensor, hi: torch.Tensor) -> float:
    return float(((y >= lo) & (y <= hi)).float().mean())


# a "distribution" with no width: 100 copies of the same number
point = xva[:, -1:].expand(-1, H)
print(f"naive point forecast     MAE  {mae(yva, point):.4f}")
print(
    f"same forecast as samples CRPS {crps(yva, point.unsqueeze(1).expand(-1, 100, -1)):.4f}   <- identical"
)

## 1.3 The free baseline (naming it, again)

`00_forecasting_baselines` named its point baseline before training anything: on this
series that is seasonal naive, repeat yesterday (validation MAE 0.2906), with plain naive,
repeat the last half-hour, far behind (0.7362). Same rule here, and the probabilistic
baseline is almost as cheap as the point one.

Take the naive forecast — repeat the last value — and ask how wrong it was, historically,
at each horizon step. Those residuals *are* an empirical predictive distribution: resample
them and you have 100 sample paths, no training, no parameters.

In [ ]:
residuals = ytr - xtr[:, -1:]  # (N, H): how wrong "repeat the last value" was, on train

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(range(1, H + 1), residuals.std(0), marker="o", ms=3)
ax.set_xlabel("hours ahead")
ax.set_ylabel("residual std")
ax.set_title("uncertainty grows with horizon — measured, not assumed")
plt.tight_layout()

draw = torch.randint(0, len(residuals), (len(xva), 100))
naive_samples = xva[:, -1:].unsqueeze(1) + residuals[draw]  # (N, 100, H)

scores: dict[str, dict] = {}


def report(
    name: str, samples: torch.Tensor, median: torch.Tensor, secs: float, params: int
) -> None:
    lo, hi = samples.quantile(0.1, dim=1), samples.quantile(0.9, dim=1)
    scores[name] = {
        "mae": mae(yva, median),
        "crps": crps(yva, samples),
        "cov80": coverage(yva, lo, hi),
        "secs": secs,
        "params": params,
    }
    r = scores[name]
    print(
        f"{name:24s} MAE {r['mae']:.4f}   CRPS {r['crps']:.4f}   "
        f"80% coverage {r['cov80']:.3f}   {r['params']:>7,} params   {r['secs']:5.1f}s"
    )


report("naive + residuals", naive_samples, naive_samples.median(1).values, 0.0, 0)

Note what the picture already told us before any model was trained: the spread at
+1 half-hour is small and at +1 day is nearly **six times larger** (measured: 5.7x —
`residuals.std(0)[-1] / residuals.std(0)[0]`). Any forecast whose interval width is
constant across the horizon is wrong on its face.

## 1.4 A Gaussian head, trained with NLL

The smallest possible change to the GRU from `00_forecasting_baselines`: instead of one
number per horizon step, output **two** — a mean and a standard deviation — and train by maximum
likelihood instead of MSE.

$$-\log p(y \mid \mu, \sigma) = \frac{(y-\mu)^2}{2\sigma^2} + \log \sigma + \text{const}$$

Read the loss rather than just importing it. The first term is MSE **divided by the
variance the model chose** — so the model can reduce its loss on hard examples by
admitting they are hard. The `log σ` term is what stops it declaring everything hard:
uncertainty has to be paid for. Together they force an honest answer, and this is why
MSE is a special case (fix `σ = 1` and only the first term survives).

Two implementation details that are not optional. Predict `log σ` and exponentiate, so
`σ > 0` is structural rather than hoped for; and clamp it, because early in training an
unlucky batch can drive `σ` to zero and the gradient to infinity.

In [ ]:
class GRUTrunk(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.rnn = nn.GRU(1, hidden, batch_first=True)
        self.hidden = hidden

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))
        return out[:, -1, :]


class GaussianHead(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.trunk = GRUTrunk(hidden)
        self.mean = nn.Linear(hidden, H)
        self.log_sigma = nn.Linear(hidden, H)

    def forward(self, x):
        h = self.trunk(x)
        return self.mean(h), self.log_sigma(h).clamp(-7, 7).exp()


def train(model, loss_fn, epochs=EPOCHS, batch_size=64, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
    t0 = time.time()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model, xb, yb).backward()
            opt.step()
    model.eval()
    return time.time() - t0


def gaussian_nll(model, x, y):
    mean, sd = model(x)
    return nn.functional.gaussian_nll_loss(mean, y, sd**2)


gaussian = GaussianHead()
secs = train(gaussian, gaussian_nll)
with torch.no_grad():
    g_mu, g_sigma = gaussian(xva)
g_samples = g_mu.unsqueeze(1) + g_sigma.unsqueeze(1) * torch.randn(len(g_mu), 100, H)
report(
    "GRU + gaussian",
    g_samples,
    g_mu,
    secs,
    sum(p.numel() for p in gaussian.parameters()),
)

print(
    f"\nlearned sigma: +30min {g_sigma[:, 0].mean():.3f}   +6h {g_sigma[:, 11].mean():.3f}   +24h {g_sigma[:, -1].mean():.3f}"
)

Nobody told the model how wide its intervals should be at each horizon step. It read that
off the data, the same way the residual plot in 1.3 did — and it now says something per
*window*, not just per horizon step: quiet stretches get narrow intervals, volatile ones
get wide.

The width is not free. As a point forecast, the mean of the Gaussian head has a validation
MAE of 0.2861: less than half the naive forecast's 0.7362, but behind the MSE-trained
GRU(64) of `00_forecasting_baselines` (0.2529, the same trunk, window and number of epochs)
and only just ahead of seasonal naive (0.2906). A trunk trained for the likelihood of a
whole distribution spends part of itself on the width, and the mean pays for it.

## 1.5 But is it *calibrated*?

A model can produce beautiful intervals that are simply the wrong size. The check is
embarrassingly direct: an 80% interval should contain the truth 80% of the time. Count.

In [ ]:
levels = torch.tensor([0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
empirical = []
for level in levels:
    z = torch.distributions.Normal(0, 1).icdf(0.5 + level / 2)
    empirical.append(coverage(yva, g_mu - z * g_sigma, g_mu + z * g_sigma))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0.4, 1], [0.4, 1], "k--", lw=1, label="perfectly calibrated")
ax.plot(levels, empirical, marker="o", label="GRU + gaussian")
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage")
ax.legend(fontsize=8)
plt.tight_layout()

for level, emp in zip(levels, empirical):
    flag = (
        "too wide"
        if emp > level + 0.02
        else ("too narrow" if emp < level - 0.02 else "ok")
    )
    print(f"nominal {float(level):.0%}  ->  empirical {emp:.1%}   {flag}")

Four of six levels sit close to the diagonal — within a point or two, which is close to
what 361 validation windows can resolve. Two are flagged **too narrow**: 50% (empirical
48.0%) and 90% (87.9%). The level this notebook actually keeps score by, 80%, lands at
78.5% — inside the "ok" band, and consistent with the `|80-cov| = 0.031` this model gets
in the full table in 1.8. That is worth sitting with for the opposite reason a bad
number would be: eyeballing "the curve looks calibrated" would still have missed that
two specific levels are off, so the discipline is to check the level you actually
report, not the general shape of the curve — even on the run where that level happens
to come out fine.

Under-coverage — what both flagged levels show here — is the less friendly failure than
over-coverage: the model is overconfident rather than cautious, and if somebody sizes a
safety margin off one of these "narrow" intervals they get less protection than the
number promised.

Calibration is a property you must measure separately from accuracy. CRPS folds the two
together into one number, which makes it a good ranking metric and a poor diagnostic —
so report both, always.

## 1.6 A quantile head: dropping the Gaussian assumption

Why should the future be symmetric and bell-shaped? Predict the quantiles directly and
find out. The tool is the **pinball loss** (also called quantile loss): for the level
`q`, penalise being too low with weight `q` and too high with weight `1 - q`.

$$L_q(y, \hat y) = \max\big(q\,(y - \hat y),\ (q - 1)(y - \hat y)\big)$$

At `q = 0.5` that is MAE, and its minimiser is the median. At `q = 0.9` under-prediction
costs nine times what over-prediction does, so the minimiser sits where 90% of the mass
is below it. Nine levels, nine outputs per horizon step, no distributional assumption
anywhere.

In [ ]:
QUANTILES = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])


def pinball(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # pred (B, Q, H), y (B, H)
    err = y.unsqueeze(1) - pred
    q = QUANTILES.view(1, -1, 1)
    return torch.maximum(q * err, (q - 1) * err).mean()


def sample_from_quantiles(pred: torch.Tensor, n: int = 100) -> torch.Tensor:
    # (N, Q, H) -> (N, n, H) by inverse-transform sampling of the piecewise-linear quantile function
    srt = pred.sort(dim=1).values
    u = torch.rand(pred.shape[0], n, pred.shape[2]) * (len(QUANTILES) - 1)
    idx = u.floor().long().clamp(max=len(QUANTILES) - 2)
    grid = srt.unsqueeze(1).expand(-1, n, -1, -1)
    lo = torch.gather(grid, 2, idx.unsqueeze(2)).squeeze(2)
    hi = torch.gather(grid, 2, (idx + 1).unsqueeze(2)).squeeze(2)
    return lo + (hi - lo) * (u - idx)


class QuantileHead(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.trunk = GRUTrunk(hidden)
        self.out = nn.Linear(hidden, H * len(QUANTILES))

    def forward(self, x):
        return self.out(self.trunk(x)).reshape(-1, len(QUANTILES), H)


quantile_gru = QuantileHead()
secs = train(quantile_gru, lambda m, x, y: pinball(m(x), y))
with torch.no_grad():
    q_pred = quantile_gru(xva)
report(
    "GRU + quantiles",
    sample_from_quantiles(q_pred),
    q_pred[:, 4, :],
    secs,
    sum(p.numel() for p in quantile_gru.parameters()),
)

crossing = float((q_pred[:, 1:, :] < q_pred[:, :-1, :]).float().mean())
print(
    f"\nquantile crossing (q90 predicted below q80, etc.): {crossing:.2%} of all pairs"
)

**Quantile crossing** is the price of dropping the assumption. Nine independent linear
outputs have nothing forcing them into order, so sometimes the model claims a 90th
percentile below its own 80th — which is not a distribution at all. `sort()` patches it,
and everybody does that, but be clear that sorting is a repair applied after the fact,
not a model that could not produce the error.

## 1.7 The control: the same head on a linear model

`00_forecasting_baselines` measured a linear layer ahead of the GRU on this series
(validation MAE 0.2381 against 0.2529) at about a hundredth of the training time. That
result has to be re-run here, or the rest of this notebook is measuring the wrong thing.
Same pinball loss, same nine quantiles, `nn.Linear` instead of a GRU.

In [ ]:
class LinearQuantile(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Linear(W, H * len(QUANTILES))

    def forward(self, x):
        return self.out(x).reshape(-1, len(QUANTILES), H)


linear_q = LinearQuantile()
secs = train(linear_q, lambda m, x, y: pinball(m(x), y))
with torch.no_grad():
    lq_pred = linear_q(xva)
report(
    "linear + quantiles",
    sample_from_quantiles(lq_pred),
    lq_pred[:, 4, :],
    secs,
    sum(p.numel() for p in linear_q.parameters()),
)

print(
    f"\nquantile crossing: {float((lq_pred[:, 1:, :] < lq_pred[:, :-1, :]).float().mean()):.2%}"
)

Two things at once, and they pull in opposite directions.

It is **the best CRPS in the table so far** (0.1737) and it trains in under a second.
The ordering `00_forecasting_baselines` measured for point forecasts survives the change
of loss: with the same pinball loss and the same nine quantiles, the linear model is ahead
of the GRU again.

And it crosses its own quantiles on **15.5% of pairs — more than three times the GRU
version's 4.6%.** With no shared trunk and no nonlinearity, each of the 432 outputs (9
quantiles x 48 horizon steps) is its own independent regression on the same 192 inputs;
nothing whatsoever couples them, so there is comparatively little stopping them from
disagreeing about order. The GRU version shares a trunk across all its outputs, which
is presumably why its crossing rate is lower without anything explicitly enforcing
order there either. The model with the best score in the table so far is also the one
that is most often not a valid distribution until you sort it.

Hold both facts. Picking by the metric alone would have hidden the second one entirely.

## 1.8 The table

Four columns, and they do not agree with each other. That is the useful part.

In [ ]:
print(
    f"{'model':24s} {'CRPS':>7} {'MAE':>7} {'cov80':>7} {'|80-cov|':>9} {'params':>9} {'secs':>7}"
)
print("-" * 76)
for name in sorted(scores, key=lambda k: scores[k]["crps"]):
    r = scores[name]
    print(
        f"{name:24s} {r['crps']:7.4f} {r['mae']:7.4f} {r['cov80']:7.3f} "
        f"{abs(r['cov80'] - 0.8):9.3f} {r['params']:9,} {r['secs']:7.1f}"
    )

Read it in this order.

**A width alone buys a lot of CRPS.** The naive point forecast scored CRPS 0.7362 in 1.2;
`naive + residuals` starts from the same forecast, adds the spread of its own past errors,
and scores 0.4411, 40% lower, without a better middle.

**The free baseline is not competitive here, and the reason is its middle.** Every trained
model lands between 0.17 and 0.21 against the baseline's 0.44. That baseline is built on
plain naive, which `00_forecasting_baselines` already found to be the weak point forecast
on this series: a flat line through a day that rises and falls. Measured as point
forecasts, the trained medians are much closer to the strong one: seasonal naive has a
validation MAE of 0.2906, the Gaussian head 0.2861. Naming the baseline is the discipline;
which baseline turns out to be hard to beat is a property of the data.

**The linear model wins again.** `linear + quantiles` takes both CRPS (0.1737) and MAE
(0.2268), ahead of `GRU + quantiles` (0.1840, 0.2517) with the same loss and the same
quantiles, in under a second against the GRU's tens of seconds. That is the ordering
`00_forecasting_baselines` measured for point forecasts (validation MAE 0.2381 against
0.2529), and it survives the change from one number to nine quantiles. Every row here is a
single seed; there, three seeds put the seed spread of both models near 0.006 to 0.007 MAE,
and the gaps in this table are wider than that.

**The `|80-cov|` column ranks the models the other way round.** The CRPS winner is the
worst-calibrated model in the table: `linear + quantiles` covers the truth with its 80%
interval only 54.1% of the time (|80-cov| = 0.259), more than double the next worst,
`GRU + quantiles` at 0.122. `GRU + gaussian`, last of the trained models on CRPS, is the
best calibrated (0.031), ahead even of the free baseline (0.078). Whether you care depends
entirely on what the forecast is for: CRPS if you are choosing a model, calibration if
somebody is going to size a threshold or a safety margin off your intervals. Pick the
metric from the decision, not from the leaderboard.

In [ ]:
picks = [40, 140, 240]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 4.4), sharey=True)
shown = {
    "GRU + gaussian": g_samples,
    "GRU + quantiles": sample_from_quantiles(q_pred),
    "linear + quantiles": sample_from_quantiles(lq_pred),
}
colors = {
    "GRU + gaussian": "tab:blue",
    "GRU + quantiles": "tab:orange",
    "linear + quantiles": "tab:green",
}

for ax, i in zip(axes, picks):
    ax.plot(range(-2 * DAY, 0), xva[i, -2 * DAY :], color="0.6", lw=1)
    ax.plot(range(H), yva[i], color="k", lw=1.8, label="truth")
    for name, s in shown.items():
        lo, hi = s[i].quantile(0.1, dim=0), s[i].quantile(0.9, dim=0)
        ax.fill_between(range(H), lo, hi, alpha=0.18, color=colors[name])
        ax.plot(range(H), s[i].median(0).values, lw=1.2, color=colors[name], label=name)
    ax.axvline(0, color="0.85")
axes[0].set_ylabel("scaled demand")
axes[-1].legend(fontsize=7)
fig.suptitle("80% intervals on three validation windows")
plt.tight_layout()

The medians are close to each other; the *widths* differ visibly. The linear quantile band
is the narrowest and the Gaussian band the widest, the same order as the coverage column.
That is the whole notebook in one picture: these models disagree more about how sure they
are than about what will happen, and only one of the two disagreements shows up in an MAE
column.

## What to take from this

- A point forecast cannot answer a question about risk, and risk is usually the question.
- CRPS scores a distribution and reduces to MAE for a point forecast, so one table holds
  both. Report calibration next to it — CRPS alone will not tell you your 80% intervals
  cover the truth 54% of the time.
- A distributional head is a small change: same trunk, two outputs instead of one. It is
  not free: the Gaussian head's mean gave up 0.033 MAE against the MSE-trained GRU of
  `00_forecasting_baselines`.
- Gaussian assumes a shape and gets coherence for free; quantiles assume nothing and can
  cross. Neither is the right default — pick from what you know about the data.
- Reproduce the baseline inside every new framing. The linear model led in
  `00_forecasting_baselines` and leads again here on CRPS and MAE; re-running it is what
  tells you whether a new model beat the standing champion or a straw man.
- The best score and the best calibration belong to different models in this table. Which
  one you ship is a question about the decision the forecast feeds.

## Exercises

**E1 — a stronger free baseline.** Build `seasonal naive + residuals`: repeat the last day,
`xtr[:, -DAY:]`, and resample its training residuals the way 1.3 does for plain naive.
Before running it, predict where it lands in the table of 1.8. Does any trained model
still beat it on CRPS, and on calibration?

**E2 — calibrate afterwards.** Take the Gaussian model's validation residuals, find the
constant `c` that makes `c * sigma` give exactly 80% coverage, and apply it to the test
split. Does a one-parameter recalibration hold up out of sample? This is conformal
prediction's core idea in three lines.

**E3 — fix the crossing.** Make `LinearQuantile` output the lowest quantile plus eight
non-negative increments (`softplus`), so the quantiles are monotone by construction.
Does forcing coherence cost accuracy?

**E4 — a fatter tail.** Swap the Gaussian head for a Student-t with a learned degrees-of-
freedom parameter (`torch.distributions.StudentT`). Where in the reliability curve does
it change things — the middle, or the 95% level?

**E5 — the honest question.** Every number in this notebook is validation. Re-score the
whole table on the test split. Given what `00_forecasting_baselines` 0.8 found on the same
test year (the ranking held, and the GRU's lead over seasonal naive shrank from 13% to 3%),
predict the answer before you run it: which rows move most, and does the CRPS ranking
survive?